In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体（2-単体）の高速生成
# ==========================================
@njit
def build_simplicial_complex(N, m):
    """
    BAモデルを構築しつつ、同時に形成される2-単体（三角形=面）を抽出する
    """
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)

    # 面（2-単体）の最大推定数
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx = 0
    target_idx = 0
    face_idx = 0

    # 初期完全グラフ (m個のノード)
    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    # 初期完全グラフ内の面を登録
    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2
        face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    # 優先的選択による成長と面の検出
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        # 新規ノードiが選んだノード群の間でエッジがあれば、面(2-単体)が形成される
        for j in range(m):
            for k in range(j + 1, m):
                node_a = chosen[j]
                node_b = chosen[k]
                # node_a と node_b が繋がっているか簡易チェック
                # (BAモデルでは、古いノード番号ほど繋がりやすい)
                faces[face_idx, 0] = i
                faces[face_idx, 1] = node_a
                faces[face_idx, 2] = node_b
                face_idx += 1 # 単純化のため、今回はトライアド候補を全て面とみなす(Flag Complex近似)

        # エッジの追加
        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

# ==========================================
# 2. フラストレーション計算と面の破裂 (Numba JIT)
# ==========================================
@njit
def evaluate_faces(X, faces, active_faces, theta):
    """
    各2-単体内部の情報勾配(フラストレーション)を計算し、閾値を超えたら破裂させる
    """
    ruptured_count = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            max_X = max(X[i], X[j], X[k])
            min_X = min(X[i], X[j], X[k])
            frustration = max_X - min_X

            # フラストレーションが閾値(許容量)を超えたら面が破裂
            if frustration > theta:
                active_faces[f] = False
                ruptured_count += 1
    return ruptured_count

# ==========================================
# 3. シミュレーション関数 (Phase 0 ~ Phase 1)
# ==========================================
def run_simplicial_simulation(N, m, params):
    dt = params['dt']
    T1 = params['T1'] # Phase 0: Burn-in終了
    T2 = params['T2'] # Phase 1a: フェイク暴露終了
    T3 = params['T3'] # Phase 1b: ヒステリシス確認終了
    steps = T3

    # ネットワーク構築
    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))

    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data
    gc.collect()

    # ★初期値依存を排除するための乱数初期化
    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    # 記録用リスト
    history_betti_1 = []

    # Euler-Poincare公式によるBetti 1 (穴) のベースライン
    # Δβ1 = 破裂した面の数 (初期状態の穴の増加を0とする)
    current_delta_betti_1 = 0

    for step in tqdm(range(steps), desc=f"Simulating N={N}", leave=False):
        # フェーズ管理
        if step == T1:
            # Phase 1a: ハブへのフェイク投下
            b[hub_node] = 0.9
        elif step == T2:
            # Phase 1b: フェイク除去（ヒステリシス検証）
            b[hub_node] = 0.0

        # 平均場(P)の計算とSDEの更新
        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        # 2-単体のフラストレーション評価と破裂判定
        # (※ペナルティにより、一度破裂した面 active_faces=False は二度と復活しない)
        new_ruptures = evaluate_faces(X, faces, active_faces, params['theta'])
        current_delta_betti_1 += new_ruptures

        history_betti_1.append(current_delta_betti_1 / max(1, num_faces)) # 穴の割合として正規化

    return history_betti_1

# ==========================================
# 4. メイン実行スクリプト
# ==========================================
def main():
    # スケール設定 (1000 から 1000万まで)
    N_list = [1000, 10000, 100000, 1000000, 10000000]

    params = {
        'dt': 0.1,
        'T1': 50,      # Burn-in
        'T2': 150,     # Fake exposure
        'T3': 250,     # Cooling / Hysteresis check
        'epsilon': 0.05, # 初期状態の乱数幅
        'sigma': 0.01,   # 認知ノイズ
        'theta': 0.3     # 面の破裂閾値 (フラストレーション許容量)
    }

    output_dir = "Simplicial_TMFG_Results"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 位相的崩壊シミュレーション開始...")
            betti_1_history = run_simplicial_simulation(N, m=3, params=params)

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['T3']),
                'Delta_Betti_1_Ratio': betti_1_history
            })
            csv_path = os.path.join(output_dir, f'result_N{N}.csv')
            df.to_csv(csv_path, index=False)

            # グラフの出力
            plt.figure(figsize=(10, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')

            # フェーズの背景色設定
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in (Null Model)')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Fake News Exposed')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Fake Removed (Hysteresis)')

            plt.title(f'Topological Phase Transition \& Hysteresis (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Proportion of Ruptured Faces ($\Delta \\beta_1 / |F|$)')
            plt.legend(loc='lower right')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'plot_N{N}.png'), dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({
                'N': N,
                'Final_Betti_1_Ratio': betti_1_history[-1]
            })

            gc.collect() # メモリ解放

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。ここまでのデータでZIP化へ移行します。")
            break

    # まとめグラフとZIP化
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'summary.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_Betti_1_Ratio'], 'k-o', linewidth=2, markersize=8)
        plt.xscale('log')
        plt.title('Asymptotic Topological Collapse across Macro-Scales')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Final Topological Holes Proportion')
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()

        zip_filename = 'Simplicial_TMFG_Hypothesis1.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:199: SyntaxWarning: invalid escape sequence '\D'
<>:206: SyntaxWarning: invalid escape sequence '\&'
<>:208: SyntaxWarning: invalid escape sequence '\D'
<>:199: SyntaxWarning: invalid escape sequence '\D'
<>:206: SyntaxWarning: invalid escape sequence '\&'
<>:208: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_4461/3721172025.py:199: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
/tmp/ipykernel_4461/3721172025.py:206: SyntaxWarning: invalid escape sequence '\&'
  plt.title(f'Topological Phase Transition \& Hysteresis (N={N:,})')
/tmp/ipykernel_4461/3721172025.py:208: SyntaxWarning: invalid escape sequence '\D'
  plt.ylabel('Proportion of Ruptured Faces ($\Delta \\beta_1 / |F|$)')



[1,000 Nodes] 位相的崩壊シミュレーション開始...



[10,000 Nodes] 位相的崩壊シミュレーション開始...



[100,000 Nodes] 位相的崩壊シミュレーション開始...



[1,000,000 Nodes] 位相的崩壊シミュレーション開始...



[10,000,000 Nodes] 位相的崩壊シミュレーション開始...



🎉 計算完了！結果は 'Simplicial_TMFG_Hypothesis1.zip' に出力されました。


In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m):
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. シミュレーション関数 (検証指標の追加)
# ==========================================
def run_simulation_with_empirical_validation(N, m, params):
    dt, T1, T2, T3 = params['dt'], params['T1'], params['T2'], params['T3']

    row, col, faces = build_simplicial_complex(N, m)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data
    gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []
    history_ks_stat = []  # 分布の定常性収束を示すKS統計量
    X_snapshots = {}      # 自己組織化証明用スナップショット

    current_delta_betti_1 = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T3), desc=f"Simulating N={N}", leave=False):
        if step == T1: b[hub_node] = 0.9
        elif step == T2: b[hub_node] = 0.0

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        current_delta_betti_1 += evaluate_faces(X, faces, active_faces, params['theta'])
        history_betti_1.append(current_delta_betti_1 / max(1, num_faces))

        # Burn-inフェーズ(T1まで)で実証的妥当性の評価計算
        if step < T1:
            if step % 10 == 0:
                # 前ステップとの分布の差をKS検定で計算（0に近づけば定常状態=自己組織化完了）
                ks_stat, _ = stats.ks_2samp(X[::100], X_prev[::100]) # 高速化のためサンプリング
                history_ks_stat.append((step, ks_stat))
                X_prev = np.copy(X)

            if step in [0, int(T1/2), T1-1]:
                X_snapshots[step] = np.copy(X)

    return history_betti_1, history_ks_stat, X_snapshots

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]
    params = {'dt': 0.1, 'T1': 100, 'T2': 200, 'T3': 300, 'epsilon': 0.05, 'sigma': 0.02, 'theta': 0.25}

    output_dir = "Empirical_Validation_Results"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 自己組織化と位相的崩壊の計算開始...")
            betti_1_hist, ks_hist, snapshots = run_simulation_with_empirical_validation(N, 3, params)

            # --- グラフ1: KS統計量による自己組織化の証明 ---
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary Distribution (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'KS_Convergence_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ2: 状態分布の進化 (一様分布からHeavy-tailへ) ---
            plt.figure(figsize=(8, 5))
            for stp, x_data in snapshots.items():
                plt.hist(x_data, bins=50, alpha=0.5, density=True, label=f'Step {stp}')
            plt.title(f'Self-Organization of Conformity Distribution (N={N})')
            plt.xlabel('Conformity (X)')
            plt.ylabel('Density')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Distribution_Evolution_N{N}.png'), dpi=300)
            plt.close()

            # --- グラフ3: 位相的崩壊 (Betti 1) ---
            df = pd.DataFrame({'TimeStep': np.arange(params['T3']), 'Delta_Betti_1_Ratio': betti_1_hist})
            df.to_csv(os.path.join(output_dir, f'betti1_result_N{N}.csv'), index=False)

            plt.figure(figsize=(10, 6))
            plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')
            plt.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in (Self-Organization)')
            plt.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Fake News Exposed')
            plt.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Fake Removed (Hysteresis)')
            plt.title(f'Topological Phase Transition & Hysteresis (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Proportion of Ruptured Faces')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Topological_Collapse_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({'N': N, 'Final_Betti_1_Ratio': betti_1_hist[-1]})
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # まとめZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'summary_all_N.csv'), index=False)
        zip_filename = 'Empirical_Validation_TMFG.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:175: SyntaxWarning: invalid escape sequence '\D'
<>:175: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_4461/2427771140.py:175: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df['TimeStep'], df['Delta_Betti_1_Ratio'], 'r-', linewidth=2.5, label='Topological Holes ($\Delta \\beta_1$)')



[1,000 Nodes] 自己組織化と位相的崩壊の計算開始...



[10,000 Nodes] 自己組織化と位相的崩壊の計算開始...



[100,000 Nodes] 自己組織化と位相的崩壊の計算開始...



[1,000,000 Nodes] 自己組織化と位相的崩壊の計算開始...



🎉 計算完了！結果は 'Empirical_Validation_TMFG.zip' に出力されました。
